In [ ]:
%run ../globalvariables

In [ ]:
from pyspark.sql.functions import col, first, lpad, avg, count, lit
from pyspark.sql.functions import max as spark_max

In [ ]:
# Target gas and district widgets
dbutils.widgets.dropdown("magnitud_target", "no2", [
    "no2", "no", "nox", "pm10", "pm2_5", "o3", "so2", "co",
    "tol", "ben", "ebe", "ch4", "nmhc", "tch",
])
dbutils.widgets.dropdown("distrito", "16", [str(i) for i in range(1, 22)])
MAGNITUD_TARGET = dbutils.widgets.get("magnitud_target")
DISTRITO = dbutils.widgets.get("distrito")
DISTRITO_TAG = f"D{DISTRITO.zfill(2)}"

MAGNITUD_MAP = {
    "no2": "NO2", "no": "NO", "nox": "NOx", "pm10": "PM10", "pm2_5": "PM2.5",
    "o3": "O3", "so2": "SO2", "co": "CO", "tol": "TOL", "ben": "BEN",
    "ebe": "EBE", "ch4": "CH4", "nmhc": "NMHC", "tch": "TCH",
}
MAGNITUD_VALUE = MAGNITUD_MAP[MAGNITUD_TARGET]

In [ ]:
# District and date aggregate, single district only
mart = spark.sql(f"""
    SELECT e.cod_dis, a.fecha,
           avg(a.dato) AS valor_medio,
           CAST(count(*) AS INT) AS n_estaciones,
           d.es_festivo
    FROM {SILVER_TABLE}.aire a
    JOIN {SILVER_TABLE}.estaciones_aire e ON CAST(a.estacion AS STRING) = e.codigo_corto
    JOIN {GOLD_TABLE}.dim_fecha d ON d.fecha = a.fecha
    WHERE a.validez = 'V' AND a.magnitud = '{MAGNITUD_VALUE}' AND e.cod_dis = '{DISTRITO}'
    GROUP BY e.cod_dis, a.fecha, d.es_festivo
""")

In [ ]:
# Manual industrial locales count, this district only
NUM_FABRICAS = {
    "1": 500, "2": 160, "3": 35, "4": 60, "5": 80, "6": 120, "7": 45, "8": 450,
    "9": 90, "10": 190, "11": 550, "12": 400, "13": 380, "14": 70, "15": 320,
    "16": 220, "17": 1400, "18": 1050, "19": 850, "20": 1100, "21": 200,
}
mart = mart.withColumn("num_fabricas_distrito", lit(NUM_FABRICAS.get(DISTRITO)))

In [ ]:
# Traffic inlined from silver, this district only
def franja_case(hour_expr):
    parts = []
    for name, (start, end) in FRANJA_HORARIA.items():
        if start < end:
            parts.append(f"WHEN {hour_expr} >= {start} AND {hour_expr} < {end} THEN '{name}'")
        else:
            parts.append(f"WHEN {hour_expr} >= {start} OR {hour_expr} < {end} THEN '{name}'")
    return "CASE " + " ".join(parts) + " END"

DISTRITO_PAD = DISTRITO.zfill(2)
readings = spark.sql(f"""
    SELECT t.id, d.distrito, CAST(t.fecha AS DATE) AS fecha,
           {franja_case('hour(t.fecha)')} AS franja,
           t.intensidad, t.ocupacion, t.carga, t.vmed
    FROM {SILVER_TABLE}.trafico t
    JOIN {GOLD_TABLE}.dim_punto_trafico d ON CAST(t.id AS INT) = d.id
    WHERE t.error = 'N' AND d.distrito = '{DISTRITO_PAD}'
""")

# Per sensor rollup first
per_sensor = readings.groupBy("id", "distrito", "fecha", "franja").agg(
    avg("intensidad").alias("intensidad"),
    avg("ocupacion").alias("ocupacion"),
    avg("vmed").alias("vmed"),
    spark_max("carga").alias("carga"),
)

# Then district rollup (already single district)
per_district = per_sensor.groupBy("distrito", "fecha", "franja").agg(
    avg("intensidad").alias("intensidad_media"),
    avg("ocupacion").alias("ocupacion_media"),
    avg("vmed").alias("vmed_media"),
    spark_max("carga").alias("carga_pico"),
    count("id").alias("n_sensores"),
)

In [ ]:
# Pivot wide per franja
trafico_wide = (
    per_district.groupBy("distrito", "fecha")
    .pivot("franja", ["Manana", "Tarde", "Noche"])
    .agg(
        first("intensidad_media").alias("intensidad_media"),
        first("ocupacion_media").alias("ocupacion_media"),
        first("carga_pico").alias("carga_pico"),
        first("vmed_media").alias("vmed_media"),
    )
)

df_final = mart.withColumn("_join_key", lpad(col("cod_dis"), 2, "0"))
df_final = df_final.join(
    trafico_wide.withColumnRenamed("distrito", "_join_key"),
    on=["_join_key", "fecha"], how="left",
).drop("_join_key")

In [ ]:
# One table per gas, district and horizon
rows = df_final.count()
FEATURE_TABLE = f"features_{MAGNITUD_TARGET}_{DISTRITO_TAG}_{FORECAST_HORIZON_GAS_DAYS}"
write_ml(df_final, FEATURE_TABLE, mode="overwrite")
print(f"{FEATURE_TABLE}: {rows} rows")

# Register as feature table
FEATURE_TABLE_FQN = f"{ML_TABLE}.{FEATURE_TABLE}"
spark.sql(f"ALTER TABLE {FEATURE_TABLE_FQN} ALTER COLUMN cod_dis SET NOT NULL")
spark.sql(f"ALTER TABLE {FEATURE_TABLE_FQN} ALTER COLUMN fecha SET NOT NULL")
spark.sql(f"ALTER TABLE {FEATURE_TABLE_FQN} DROP CONSTRAINT IF EXISTS {FEATURE_TABLE}_pk")
spark.sql(f"ALTER TABLE {FEATURE_TABLE_FQN} ADD CONSTRAINT {FEATURE_TABLE}_pk PRIMARY KEY (cod_dis, fecha)")